In [34]:
import random

import numpy as np

In [35]:
def create_initial_population(num_nodes, population_size):
    initial_population = []
    for _ in range(population_size):
        initial_solution = np.random.randint(0, 2, size=num_nodes, dtype=int)
        initial_population.append(initial_solution)
    return initial_population

In [36]:
def calculate_fitness(solution, graph):
    num_nodes = len(graph)
    covered_nodes = set()
    solution_nodes = set()

    # enumerate krece od 0, meni su svi cvorovi od 1
    for i, val in enumerate(solution):
        if val == 1:
            solution_nodes.add(i + 1)

    covered_nodes.update(solution_nodes)
    for node in solution_nodes:
        covered_nodes.update(graph.get(node, []))

    n = len(covered_nodes)

    gamma_x = np.sum(solution)
    if gamma_x == 0:
        return 0.0

    fitness_value = n / num_nodes + 1 / (num_nodes * gamma_x)

    return fitness_value

In [37]:
def get_node_degrees(graph):
    degrees = {}
    for node, neighbors in graph.items():
        degrees[node] = len(neighbors)
    return degrees

In [38]:
def local_search(solution, graph, num_iters):
    best_solution = solution.copy()

    for _ in range(num_iters):
        current_solution = best_solution.copy()
        current_fitness = calculate_fitness(current_solution, graph)

        node_degrees = get_node_degrees(graph)

        if current_fitness >= 1:
            candidates = [i for i, val in enumerate(current_solution) if val == 1]
            weights = [1 / node_degrees[i + 1] for i in candidates]
        else:
            candidates = [i for i, val in enumerate(current_solution) if val == 0]
            weights = [node_degrees[i + 1] for i in candidates]

        if not candidates:
            continue

        chosen_index = random.choices(candidates, weights=weights, k=1)[0]
        current_solution[chosen_index] = 0 if current_fitness >= 1 else 1

        new_fitness = calculate_fitness(current_solution, graph)
        if new_fitness > current_fitness:
            best_solution = current_solution.copy()

    return best_solution

In [39]:
def filtering(solution, graph):
    current_fitness = calculate_fitness(solution, graph)
    if current_fitness < 1:
        return solution.copy()

    filtered_solution = solution.copy()
    dominating_node_indices = [i for i, val in enumerate(filtered_solution) if val == 1]

    for node_position in dominating_node_indices:
        filtered_solution[node_position] = 0

        new_fitness = calculate_fitness(filtered_solution, graph)
        if new_fitness < 1:
            filtered_solution[node_position] = 1

    return filtered_solution

In [40]:
def compute_covered_nodes(solution, graph):
    selected_nodes = {i + 1 for i, bit in enumerate(solution) if bit == 1}
    covered_nodes = set(selected_nodes)
    for node in selected_nodes:
        covered_nodes.update(graph.get(node, set()))
    return covered_nodes

In [42]:
def elite_inspiration(elite_sets, graph, n_core):
    if elite_sets is None or len(elite_sets) < n_core:
        return None

    elite_sets_sorted = sorted(elite_sets, key=lambda sol: int(sol.sum))
    core_candidates = elite_sets_sorted[:n_core]

    x_core = np.logical_and.reduce(core_candidates).astype(np.uint8)

    x_best = core_candidates[0]
    target_size = int(x_best.sum())
    current_size = int(x_core.sum())

    node_degrees = get_node_degrees(graph)

    while (target_size - current_size) > 1:
        zero_indices = np.where(x_core == 0)[0]
        if zero_indices.size == 0:
            break

        chosen_index = max(zero_indices, key=lambda i: node_degrees.get(i + 1, 0))
        x_core[chosen_index] = 1
        current_size += 1

    return x_core